# HIXL 性能分析方法

本节重点介绍 HIXL 性能问题的分析方法，涵盖关键日志分析和 Profiling 性能数据采集与解析。学完本节后，你将能够结合日志和 Profiling 数据定位 HIXL 应用开发过程中的性能瓶颈，为后续优化提供依据。

本节学习大纲如下：

- HIXL 关键性能日志分析
- HIXL Profiling 采集方法
- HIXL Profiling 数据解析


---

## 1. HIXL 关键性能日志分析

HIXL 底层依赖 HCCL 通信库，性能统计日志通过 HCCL 输出。通过分析统计日志，可快速定位性能瓶颈在建链阶段还是传输阶段，并确定当前使用的传输模式。

> 注意：下述的性能统计信息通过心跳机制每 90s 打印一次，运行时间太短可能会获取不到相关性能日志。


### 1.1 快速上手：提取关键日志

使用以下命令提取 HIXL 性能统计日志：

```bash
grep -R "Connect statistic info\|Connect hccl detail\|Direct transfer statistic info\|Buffer transfer statistic info\|Fabric mem transfer statistic info" ~/ascend/log
```

若已定位某条链路或某个 channel 存疑，可进一步按 channel 过滤：

```bash
grep -R "Connect statistic info\|Connect hccl detail\|Direct transfer statistic info\|Buffer transfer statistic info\|Fabric mem transfer statistic info" ~/ascend/log | grep "channel:"
```

**排查原则**：建议将同一时间窗口内的三类日志放在一起分析——建链统计、HCCL 建链细节、当前模式对应的传输统计。


### 1.2 排查思路：先判断慢在建链还是传输

定位性能问题时，首先查看 `Connect statistic info` 日志：

- 若建链统计耗时明显偏高，优先分析建链问题。
- 若建链正常，再查看传输统计。

此顺序至关重要，许多看似"传输慢"的问题，根因往往在建链阶段就已异常。


### 1.3 建链统计解读

三种传输模式均需先查看建链统计。

**建链总览日志模板**：

```text
Connect statistic info[channel:<channel_id>, total times:<N>, max cost:<X> us, avg cost:<Y> us,
tcp times:<N1>, max cost:<X1> us, avg cost:<Y1> us,
hccl times:<N2>, max cost:<X2> us, avg cost:<Y2> us,
other avg cost:<Y3> us].
```

**字段含义**：

| 字段 | 含义 | 分析建议 |
| --- | --- | --- |
| `total` | 整个建链流程总耗时 | 若偏高，先确认是否整体建链偏慢 |
| `tcp` | TCP 建立 socket 耗时 | 若偏高，优先排查控制面 socket 建立、网络连通性或双端启动时序 |
| `hccl` | HCCL 建链阶段总耗时 | 若偏高，需进一步查看 HCCL 建链细节日志 |
| `other` | 其余开销（按 total - tcp - hccl 计算） | 若偏高，慢点更可能在 connect 握手、channel 建立、FabricMem 导入共享内存等外围流程 |

---

**HCCL 建链细节日志**（直传模式和 Buffer 模式通常有此日志，FabricMem 模式下该部分一般为 0）：

```text
Connect hccl detail[channel:<channel_id>, init times:<N1>, max cost:<X1> us, avg cost:<Y1> us,
bind times:<N2>, max cost:<X2> us, avg cost:<Y2> us,
prepare times:<N3>, max cost:<X3> us, avg cost:<Y3> us].
```

**字段含义**：

| 字段 | 对应接口 | 分析建议 |
| --- | --- | --- |
| `init` | `HcclCommInitClusterInfoMemConfig` | 若偏高，优先排查 ranktable、设备信息、HCCL 初始化阶段 |
| `bind` | `HcclCommBindMem` | 若偏高，优先排查建链前注册内存规模、内存类型和绑定数量 |
| `prepare` | `HcclCommPrepare` | 若偏高，优先排查 HCCL 真正建链阶段，常与双端时序、链路不一致、超时问题关联 |

---

**三种模式建链关注点**：

| 模式 | 建链分析建议 |
| --- | --- |
| 直传 | 先看 `total/tcp/hccl`，再结合 `Connect hccl detail` 找慢点 |
| 中转（Buffer） | 建链过程和直传类似，同样先看 `total/tcp/hccl` 与 HCCL 细节 |
| FabricMem | 重点看 `total/tcp/other`；`hccl` 一般为 0 |


### 1.4 传输统计解读

确认建链无明显异常后，根据当前传输模式查看对应的传输统计日志。


**直传模式**

日志模板：

```text
Direct transfer statistic info[channel:<channel_id>, transfer times:<N>, total size:<S> kBytes,
avg size:<L> kBytes, avg bandwidth:<B> GB/s, max cost:<X> us, avg cost:<Y> us].
```

**字段含义**：

| 字段 | 含义 | 分析建议 |
| --- | --- | --- |
| `transfer` | 直传模式单次传输整体耗时 | 若偏高，问题更可能在直传数据面本身 |
| `total size` | 统计窗口内累计传输量 | 按十进制 kBytes 打印 |
| `avg size` | 平均每个 `TransferOpDesc` 的字节量 | 若很小但 `avg cost` 不低，常见于小包场景，固定开销放大 |
| `avg bandwidth` | 平均带宽 | 若偏低，可与其他模式横向比较单位时间内实际搬运数据量 |


**中转（Buffer）模式**

日志模板：

```text
Buffer transfer statistic info[channel:<channel_id>, transfer times:<N>, total size:<S> kBytes,
avg size:<L> kBytes, avg bandwidth:<B> GB/s, max cost:<X> us, avg cost:<Y> us,
client copy times:<N1>, max cost:<X1> us, avg cost:<Y1> us,
server comm times:<N2>, max cost:<X2> us, avg cost:<Y2> us,
server copy times:<N3>, max cost:<X3> us, avg cost:<Y3> us].
```

**字段含义**：

| 字段 | 含义 | 分析建议 |
| --- | --- | --- |
| `transfer` | Buffer 模式整体传输耗时 | 若偏高，先确认整体确实慢在 Buffer 路径 |
| `client copy` | client 侧 copy 耗时 | 若偏高，优先排查 client 侧地址转换、分批或 copy 开销 |
| `server comm` | server 侧 D2D 或通信阶段耗时 | 若偏高，优先排查 server 侧通信阶段 |
| `server copy` | server 侧数据搬运耗时 | 若偏高，优先排查 server 侧数据搬运阶段 |

**排查建议**：分析 Buffer 模式时，建议同时观察链路两端的统计日志。


**FabricMem 模式**

日志模板：

```text
Fabric mem transfer statistic info[channel:<channel_id>, transfer times:<N>, total size:<S> kBytes,
avg size:<L> kBytes, avg bandwidth:<B> GB/s, max cost:<X> us, avg cost:<Y> us,
real copy times:<N1>, max cost:<X1> us, avg cost:<Y1> us].
```

**字段含义**：

| 字段 | 含义 | 分析建议 |
| --- | --- | --- |
| `transfer` | FabricMem 模式整体传输耗时 | 若偏高且 `real copy` 也高，优先怀疑数据拷贝本身慢 |
| `real copy` | 真实 copy 阶段耗时 | 若不高但 `transfer` 高，慢点更可能在地址转换、任务拆分、流同步等外围逻辑 |


### 1.5 常用排查套路速查

**怀疑建链慢**：

1. 查看 `Connect statistic info`。
2. 比较 `total`、`tcp`、`hccl`、`other`。
3. 若 `hccl` 高，查看 `Connect hccl detail`。
4. 若为 FabricMem，重点分析 `tcp` 和 `other`。

---

**怀疑直传模式慢**：

1. 确认当前模式为直传模式。
2. 查看 `Direct transfer statistic info`。
3. 若 `avg cost` 明显偏高，问题大概率落在直传数据面。

---

**怀疑 Buffer 模式慢**：

1. 确认当前传输走了 Buffer 路径。
2. 查看 `Buffer transfer statistic info`。
3. 按 `transfer → client copy → server comm → server copy` 顺序定位慢点。

---

**怀疑 FabricMem 模式慢**：

1. 确认当前模式为 FabricMem。
2. 查看 `Fabric mem transfer statistic info`。
3. 比较 `transfer` 和 `real copy`，判断慢在真实 copy 还是外围流程。


### 1.6 判断异常的注意事项

统计日志可快速缩小排查范围，但不宜仅凭一条日志直接下结论。建议结合传输大小、调用频次和日志级别综合判断。

**通常不建议直接判定为异常的情况**：

- 建链耗时未超过 1 秒。
- `avg size` 小于 16 KB（小包场景，固定开销易放大）。
- `avg size` ≥ 16 KB，且传输带宽不低于 5 GB/s。

**其他注意点**：

- 若日志级别为 `INFO`，耗时统计可能不够稳定，建议切到 `ERROR` 再观察。
- `avg size` 和 `avg bandwidth` 需结合业务实际传输模型判断，不宜脱离报文大小单独下结论。


## 2. HIXL Profiling 采集方法

通过 Profiling 工具可采集更细粒度的性能数据，包括 CANN 层接口耗时和底层 NPU 任务执行情况。本节介绍 ACL C++ 接口的 Profiling 采集方法。

整体流程如下：

```
配置 Profiling 参数
       ↓
aclprofStart / with torch_npu.profiler.profile
       ↓
执行目标代码（传输、推理等）
       ↓
aclprofStop / 退出 with 块
       ↓
（ACL 方式需额外执行）msprof --export=on --output=./
       ↓
chrome://tracing 或 MindStudio Insight 查看结果
```

### 2.1 ACL C++ Profiling 接口

在传输接口调用前后添加 Profiling 采集代码，示例如下：

```cpp
// 1. 初始化 ACL 运行时
auto acl_init_ret = aclInit(nullptr);
if (acl_init_ret != ACL_SUCCESS) {
    printf("[ERROR] Acl init failed\n");
    return -1;
}

// 2. 初始化 Profiling，设置数据落盘路径
const char *acl_prof_path = "./output";
aclprofInit(acl_prof_path, strlen(acl_prof_path));

// 3. 创建 Profiling 配置
//    - device_id_list: 须根据实际环境的 Device ID 配置
//    - ACL_AICORE_ARITHMETIC_UTILIZATION: 采集 AI Core 算术利用率
//    - ACL_PROF_ACL_API | ACL_PROF_TASK_TIME: 同时采集 ACL API 调用和任务耗时
uint32_t device_id_list[1] = {1};
aclprofConfig *config = aclprofCreateConfig(
    device_id_list, 1,
    ACL_AICORE_ARITHMETIC_UTILIZATION,
    nullptr,
    ACL_PROF_ACL_API | ACL_PROF_TASK_TIME
);

// 4. （可选）设置扩展采集参数，如系统内存采集频率（单位 ms）
const char *mem_freq = "15";
auto prof_ret = aclprofSetConfig(ACL_PROF_SYS_HARDWARE_MEM_FREQ, mem_freq, strlen(mem_freq));
if (prof_ret != ACL_SUCCESS) {
    printf("[ERROR] Acl setting config failed\n");
    return -1;
}

// 5. 开始采集
aclprofStart(config);

// ===== 在此区间内执行需要分析的代码 =====
if (TransferAsync(hixl_engine, src, remote_engine, remote_addr, op_type) != 0) {
    Disconnect(hixl_engine, remote_engine);
    Finalize(hixl_engine, is_host, {handle}, {src});
    return -1;
}
// =========================================

// 6. 停止采集，销毁配置，释放 Profiling 资源（须成对调用）
aclprofStop(config);
aclprofDestroyConfig(config);
aclprofFinalize();
```

### 2.2 接口说明

各接口须**成对调用**，生命周期如下：

| 接口 | 配对接口 | 说明 |
| --- | --- | --- |
| `aclprofInit` | `aclprofFinalize` | 初始化 Profiling，设置性能数据保存路径 |
| `aclprofCreateConfig` | `aclprofDestroyConfig` | 创建 / 销毁 Profiling 配置对象 |
| `aclprofSetConfig` | — | `aclprofCreateConfig` 的扩展接口，用于设置额外采集参数 |
| `aclprofStart` | `aclprofStop` | 开始 / 停止性能数据采集 |

各接口完整参数说明请参考 [接口文档](https://www.hiascend.com/document/detail/zh/canncommercial/850/API/appdevgapi/appdevgapi_07_0000.html)。

### 2.3 数据导出

执行完用例后，在 `acl_prof_path` 指定的路径下会生成名为 `PROF_xxxxxx` 的目录。进入该目录执行：

```bash
msprof --export=on --output=./
# --output 路径支持自定义
```

执行成功后将在 `output` 目录下生成以下结构：

```
output/
├── mindstudio_profiler_log/      # 采集日志
└── mindstudio_profiler_output/
    └── msprof_xxxxx.json         # 可视化 timeline 数据
```

### 2.4 查看可视化数据

在 Chrome 浏览器中打开 `chrome://tracing`，将 `msprof_xxxxx.json` 拖入空白处即可查看性能 timeline。


## 3. HIXL Profiling 数据解析

下面以某次实际采集数据为例，说明主要性能数据的读取方式。该数据记录了一次 `hixlOpBatchWrite` 操作的完整性能快照，使用 Chrome Trace Event 格式（即 `msprof` 导出的 `msprof_xxxxx.json`），可直接拖入 `chrome://tracing` 查看，如下图：
<img src="./images/profiling.png" alt="profiling数据" width="100%">

### JSON 数据格式说明

Chrome Trace Event 格式中，每条记录为一个 JSON 对象，关键字段如下：

| 字段 | 含义 |
| --- | --- |
| `ph` | 事件类型：`M`=元数据（进程/线程命名），`X`=耗时事件（有起止），`C`=计数器（折线图），`s`/`f`=流事件（箭头，连接因果关系） |
| `pid` | 进程 ID，对应 timeline 中的一个分区（Process） |
| `tid` | 线程 ID，对应分区内的一条泳道（Track/Stream） |
| `ts` | 时间戳，单位 μs（微秒），chrome trace 会自动对齐到相对时间 |
| `dur` | 耗时，单位 μs，仅 `ph=X` 时有效 |
| `args` | 附加属性，点击 timeline 色块后在详情面板中展示 |

### Timeline 中的 Process 分区

在 `chrome://tracing` 中，本次数据展示为以下 5 个 Process 分区（均归属于 NPU 1）：

| 分区名 | 说明 |
| --- | --- |
| **CANN** | Host 侧 AscendCL API 调用序列，反映 CPU 线程的执行流，是分析入口 |
| **Ascend Hardware** | NPU 上各 Stream 的任务执行情况，按物理 Stream ID 分泳道展示 |
| **Communication** | HCCL 集合通信算子的执行记录，包含通信组名、算法类型、数据量等 |
| **AI Core Freq** | AI Core 主频随时间的变化（计数器折线），用于判断是否发生动态调频 |
| **Overlap Analysis** | 将 NPU 时间分解为 Computing / Communication / Free 三类，直观评估计算通信重叠度 |

### 实例解读：一次 hixlOpBatchWrite 调用

下面结合 `data.json` 中的真实数据，逐层说明一次完整的传输操作的调用链。以 `hixlOpBatchWrite` 开始时刻为零点，关键事件时间线如下（时间单位 μs）：

```
相对时间(μs)  所在层                 事件                              耗时(μs)
  0           CANN                  hixlOpBatchWrite                  895.78   ← 顶层入口
 +114         CANN                  OpType::BATCHSENDRECV             586.32   ← HCCL 集合通信（HOST_HCCL）
 +185         CANN                  aclrtMemcpy #1                     75.45   ← 复制第一段元数据到 Device
 +272         CANN                  aclrtMemcpy #2                     27.44   ← 复制第二段元数据
 +309         CANN                  Node@launch(batch_putAicpuKernel) 110.34   ← 准备下发 AI CPU kernel
 +335         CANN                  aclrtLaunchKernelWithHostArgs       83.53   ← 实际 kernel 下发调用
                                         ║ HostToDevice 流事件（箭头）
 +414         Ascend Hardware       batch_putAicpuKernel              324.64   ← NPU Stream 9 上执行 AI CPU kernel
 +502         CANN                  Communication@Notify_Wait         113.09   ← Host 等待 NPU 完成通知
 +503         CANN                  aclrtWaitAndResetNotify            47.64   ← 阻塞等待 notify 信号
 +573         Ascend Hardware       WRITE_VALUE_SQE（Stream 50）        0.30   ← 写入信号量（同步机制）
 +619         Ascend Hardware       WRITE_VALUE_SQE（Stream 50）        0.32   ← 写入第二个信号量
 +739         Ascend Hardware       NOTIFY_WAIT（Stream 9）             0.02   ← NPU 收到 notify，任务结束
 +739         Communication         hccl_batchPut__724_0_1             0.02   ← 集合通信完成记录
 +778         CANN                  aclrtSynchronizeStreamWithTimeout  33.90   ← Host 等待 Stream 完全同步
 +895         hixlOpBatchWrite 结束
```

**关键观察：**

1. **Host-Device 异步流水**：`aclrtLaunchKernelWithHostArgs`（+335）完成后，NPU 上的 `batch_putAicpuKernel`（+414）才开始执行，两者之间有约 80 μs 的调度延迟。此后 Host 进入 `Notify_Wait` 阶段（+502），与 NPU kernel 执行形成部分重叠。

2. **流事件（HostToDevice 箭头）**：`data.json` 中 `ph="s"/"f"` 的事件对（如 `HostToDevice1209091842000146715901951`）在 chrome://tracing 中以箭头形式展示，直观连接 Host 侧的 launch 调用与 NPU 侧的 kernel 执行，帮助追踪因果关系。

3. **AI Core Freq 稳定**：本次采集全程频率为 **1850 MHz**，未发生动态调频，因此 Ascend Hardware 层的 `Aicore Time` 字段（理论执行时间）数据可信，可作为参考。

4. **总延迟分解**：`hixlOpBatchWrite` 总耗时 ~896 μs，其中 NPU kernel 自身执行 324.64 μs，其余约 570 μs 为 Host 侧的数据准备（memcpy）、kernel launch、同步等待等开销，说明 Host 开销是该操作的主要瓶颈之一。

### Communication 层

Communication 层记录 HCCL 集合通信算子的执行信息。本例中：

```json
{
    "name": "hccl_batchPut__724_0_1",
    "args": {
        "model id": 4294967295,
        "data_type": "UINT8",
        "alg_type": "RING-RING-NHR",
        "count": 40
    }
}
```

- **通信组**：`Group hixl_7.150.3.165:26001_7.150.3.165:26000`，为本机两个进程间的点对点通信
- **算法**：`RING-RING-NHR`（分层环形算法），HCCL 根据拓扑自动选择
- **数据量**：`count=40` 的 `UINT8`，即 40 字节（这是控制信令，非实际数据 payload）
- **耗时**：0.02 μs，通信本身极短，说明此处的传输延迟主要来自 Host 侧调度和 NPU kernel 执行

### Overlap Analysis 层

Overlap Analysis 将 NPU 时间拆分为 4 类，用于评估计算与通信的重叠程度：

| 泳道 | 本例耗时(μs) | 含义 |
| --- | --- | --- |
| Computing | 324.64 | NPU 执行计算 kernel 的时间 |
| Communication | 0.02 | 通信算子的执行时间（含与 Computing 重叠部分） |
| Communication(Not Overlapped) | 0.02 | 纯通信时间（不与计算重叠） |
| Free | 0.02 | NPU 空闲/等待时间 |

本例中 Computing（324.64 μs）远大于 Communication（0.02 μs），且通信与计算几乎不重叠（`Communication ≈ Communication(Not Overlapped)`），说明该操作是**计算密集型**，通信代价可忽略。若 `Communication(Not Overlapped)` 占比较大，则说明通信成为瓶颈，需要考虑计算通信流水线优化。

### CANN 层数据

<img src="./images/cann_timeline.png" alt="cann_timeline" width="100%">

该层包含 Runtime 等 CANN 组件以及算子（Node）的耗时数据。在 timeline 中点击色块可查看接口详情，字段含义如下：

| 字段名 | 含义 |
| --- | --- |
| Title | 接口名称 |
| Start | 时间轴上的时刻点（chrome trace 自动对齐），单位 ms |
| Wall Duration | 当前接口调用的总耗时（含子调用），单位 ms |
| Self Time | 当前接口自身执行耗时（不含子调用），单位 ms |
| Mode | AscendCL API 类型：`ACL_OP`（单算子）、`ACL_MODEL`（模型接口）、`ACL_RTS`（Runtime 接口）等 |
| level | 层级标识，此处为 AscendCL 层 |

> **性能基准参考**：具体耗时与数据块大小相关，可对照 benchmark 数据（[A2](https://gitcode.com/cann/hixl/blob/master/benchmarks/A2_benchmark_performance.md) / [A3](https://gitcode.com/cann/hixl/blob/master/benchmarks/A3_benchmark_performance.md)）判断是否正常。**±10% 以内的波动属于正常抖动**。

### 底层 NPU 数据（Ascend Hardware）

<img src="./images/hardware_timeline.png" alt="hardware_timeline" width="100%">

该层展示各 Stream 上 AI 任务的调度信息，可直观判断任务调度耗时。字段含义如下：

| 字段名 | 含义 |
| --- | --- |
| Title | 组件接口名称 |
| Start | 时间轴上的时刻点，单位 ms |
| Wall Duration | 当前接口调用耗时，单位 ms |
| Task Time(us) | AI CPU 算子的 Task 任务耗时，单位 us |
| Reduce Duration(us) | ALL REDUCE 算子的集合通信时间，单位 us |
| Model Id | 模型 ID |
| Task Type | 执行该 Task 的加速器类型：`AI_CORE`、`AI_VECTOR_CORE`、`AI_CPU` 等 |
| Stream Id | Task 所处的 Stream ID。Ascend Hardware 下显示完整逻辑流 ID；右侧 timeline 各接口的 Stream Id 为物理流 ID（Physical Stream Id） |
| Task Id | Task ID |
| Subtask Id | Subtask ID |
| Aicore Time(ms) | 所有 Block 同时调度且执行时长相等时的理论执行时间，单位 ms。实际因 Block 调度时间差略有偏差，通常略小于实际值。**注意**：手动调频、动态调频（功耗超限）或 Atlas 300V/300I Pro 场景下该字段不准确，不建议参考 |
| Total Cycle | 该 Task 在 AI Core 上执行的总 cycle 数（所有 Block 累加） |
| Receive Time | Device 收到内存拷贝 Task 信息的时间，单位 us。**仅 MemcopyAsync 展示** |
| Start Time | 内存拷贝 Task 开始拷贝的时间，单位 us。**仅 MemcopyAsync 展示** |
| End Time | 内存拷贝 Task 结束拷贝的时间，单位 us。**仅 MemcopyAsync 展示** |
| size(B) | 拷贝的数据量，单位 B。**仅 MemcopyAsync 展示** |
| bandwidth(GB/s) | 拷贝带宽，单位 GB/s。**仅 MemcopyAsync 展示** |
| operation | 拷贝类型（如 host to device / device to host）。**仅 MemcopyAsync 展示** |

### 其他数据文件

导出后的 `mindstudio_profiler_output` 目录还包含以下数据文件：

| 文件 | 说明 |
| --- | --- |
| `msprof_*.json` | Timeline 总表，用于 chrome://tracing 可视化 |
| `op_summary_*.csv` | AI Core 和 AI CPU 算子执行数据 |
| `op_statistic_*.csv` | 算子调用次数及耗时统计 |
| `task_time_*.csv` | Task Scheduler 任务调度信息 |
| `fusion_op_*.csv` | 算子融合前后对比信息 |
| `api_statistic_*.csv` | CANN 层 API 执行耗时统计 |
| `step_trace_*.json` | 迭代轨迹数据（每轮迭代耗时） |

> **提示**：当 CSV 文件中出现科学计数法（如 `1.00159E+12`）时，可在 Excel 中将单元格格式设置为"数值"以正常显示；`N/A` 表示该字段在当前场景下不适用。

更多性能数据字段说明请参考 [官方文档](https://www.hiascend.com/document/detail/zh/canncommercial/850/devaids/Profiling/atlasprofiling_16_0057.html)。


---

## 课后练习

本节介绍了 HIXL 性能日志分析以及 Profiling 的采集、导出和解析方法，请根据学习内容完成以下题目进行自测。

1. （判断题）HIXL 性能统计日志通过心跳机制每 90 秒打印一次，因此运行时间过短时可能获取不到相关性能日志。

2. （判断题）在 ACL C++ Profiling 采集中，只要调用 `aclprofStart` 和 `aclprofStop` 即可，`aclprofDestroyConfig` 与 `aclprofFinalize` 可以省略。

3. （单选题）当 `Connect statistic info` 中的 `hccl` 耗时明显偏高时，下一步最适合重点查看哪类日志？
    A. `task_time_*.csv`
    B. `Connect hccl detail`
    C. `step_trace_*.json`
    D. `AI Core Freq`

4. （单选题）在 Chrome Trace Event 格式中，哪个字段表示事件耗时，且仅在 `ph = X` 时有效？
    A. `pid`
    B. `ts`
    C. `dur`
    D. `args`

5. （多选题）关于本章介绍的性能分析方法，以下哪些说法是正确的？
    A. 分析 Buffer 模式性能时，建议同时观察链路两端的统计日志
    B. FabricMem 模式下若 `transfer` 高且 `real copy` 也高，应优先怀疑真实数据拷贝本身慢
    C. CANN 层中若 Wall Duration 远大于 Self Time，往往说明主要耗时在子接口调用链上
    D. `mindstudio_profiler_log/` 目录下保存的是可直接拖入 `chrome://tracing` 的 timeline JSON 文件

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/04.03_answer.txt